# MLOps Assignment 2 - Hugging Face Fine-Tuning, W&B Tracking, and Model Deployment

**Student:** Hemant Kumar  
**Model:** `distilbert-base-cased`  
**Dataset:** UCSD Goodreads reviews by genre  

This notebook implements the required MLOps workflow: secure secrets, model fine-tuning, W&B experiment tracking, evaluation reporting, W&B artifact upload, Hugging Face Hub model publishing, and final public links.

## 1. Install compatible dependencies



In [3]:
!pip uninstall -y transformers huggingface_hub tokenizers accelerate -q
!pip install -q     transformers==4.52.4     huggingface_hub==0.33.5     tokenizers==0.21.1     accelerate==1.8.1     datasets     evaluate     wandb     scikit-learn     gdown

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.33.5 which is incompatible.


## 2. Load Kaggle Secrets and authenticate services

Before running this cell, add two Kaggle Secrets and attach them to this notebook:

- `WANDB_API_KEY`
- `HF_TOKEN`

The W&B `entity` is explicitly set to `hemantkumarsri-mlops` to ensure the run logs to the correct public W&B workspace.

In [4]:
from kaggle_secrets import UserSecretsClient
import os
import wandb
from huggingface_hub import login, create_repo

secrets = UserSecretsClient()
WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")
HF_TOKEN = secrets.get_secret("HF_TOKEN")

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["TOKENIZERS_PARALLELISM"] = "false"

wandb.login(key=WANDB_API_KEY)
login(token=HF_TOKEN)

print("Hugging Face and W&B authentication successful.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hemantkumarsri (hemantkumarsri-mlops) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face and W&B authentication successful.


## 3. Import libraries

In [5]:
from collections import defaultdict
import gzip
import json
import pickle
import random
import requests

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Torch CUDA available:", torch.cuda.is_available())

2026-05-27 16:17:51.415087: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779898671.632942      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779898671.695054      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779898672.218541      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779898672.218570      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779898672.218573      58 computation_placer.cc:177] computation placer alr

Torch CUDA available: True


## 4. Set parameters and public project links

In [6]:
model_name = "distilbert-base-cased"
max_length = 512
num_train_epochs = 3
train_batch_size = 8
learning_rate = 3e-5
cached_model_directory_name = "distilbert-goodreads-genres"
repo_id = "hemantkumarsri/distilbert-goodreads-genres"
wandb_entity = "hemantkumarsri-mlops"
wandb_project = "mlops-assignment2"
wandb_run_name = "distilbert-run-1"

github_url = "https://github.com/hemantkumarsri/mlops-assignment2"
kaggle_url = "https://www.kaggle.com/code/hemantkumarsri/mlops-assignment-2-fine-tuning-classification-k"
hf_model_url = f"https://huggingface.co/{repo_id}"
wandb_dashboard_url = f"https://wandb.ai/{wandb_entity}/{wandb_project}"

device_name = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device_name)
print("Hugging Face Model:", hf_model_url)
print("W&B Dashboard:", wandb_dashboard_url)

Device: cuda
Hugging Face Model: https://huggingface.co/hemantkumarsri/distilbert-goodreads-genres
W&B Dashboard: https://wandb.ai/hemantkumarsri-mlops/mlops-assignment2


## 5. Initialize W&B experiment tracking

This is the run that will collect hyperparameters, training loss, evaluation loss, accuracy, F1 score, and final artifacts.

In [7]:
wandb.init(
    entity=wandb_entity,
    project=wandb_project,
    name=wandb_run_name,
    config={
        "model": model_name,
        "epochs": num_train_epochs,
        "batch_size": train_batch_size,
        "learning_rate": learning_rate,
        "max_length": max_length,
        "dataset": "UCSD Goodreads Reviews by Genre",
        "platform": "Kaggle GPU",
        "github_url": github_url,
        "kaggle_url": kaggle_url,
        "huggingface_model_url": hf_model_url,
    }
)

## 6. Load and sample Goodreads review data

The dataset is downloaded directly from the UCSD Goodreads public data repository. We use eight genres and sample a small balanced dataset so that training completes reliably on Kaggle GPU.

In [8]:
genre_url_dict = {
    "poetry": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_poetry.json.gz",
    "children": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_children.json.gz",
    "comics_graphic": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_comics_graphic.json.gz",
    "fantasy_paranormal": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_fantasy_paranormal.json.gz",
    "history_biography": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_history_biography.json.gz",
    "mystery_thriller_crime": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_mystery_thriller_crime.json.gz",
    "romance": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_romance.json.gz",
    "young_adult": "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/byGenre/goodreads_reviews_young_adult.json.gz",
}

def load_reviews(url, head=10000, sample_size=1000):
    reviews = []
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()
    with gzip.open(response.raw, "rt", encoding="utf-8") as file:
        for idx, line in enumerate(file):
            d = json.loads(line)
            review = d.get("review_text", "").strip()
            if review:
                reviews.append(review)
            if head is not None and idx + 1 >= head:
                break
    return random.sample(reviews, min(sample_size, len(reviews)))

genre_reviews_dict = {}
for genre, url in genre_url_dict.items():
    print(f"Loading reviews for genre: {genre}")
    genre_reviews_dict[genre] = load_reviews(url, head=10000, sample_size=1000)

pickle.dump(genre_reviews_dict, open("genre_reviews_dict.pickle", "wb"))
print({genre: len(reviews) for genre, reviews in genre_reviews_dict.items()})

Loading reviews for genre: poetry
Loading reviews for genre: children
Loading reviews for genre: comics_graphic
Loading reviews for genre: fantasy_paranormal
Loading reviews for genre: history_biography
Loading reviews for genre: mystery_thriller_crime
Loading reviews for genre: romance
Loading reviews for genre: young_adult
{'poetry': 1000, 'children': 1000, 'comics_graphic': 1000, 'fantasy_paranormal': 1000, 'history_biography': 1000, 'mystery_thriller_crime': 1000, 'romance': 1000, 'young_adult': 1000}


## 7. Split into train and test sets

In [9]:
train_texts, train_labels = [], []
test_texts, test_labels = [], []

for genre, reviews in genre_reviews_dict.items():
    reviews = random.sample(reviews, 1000)
    for review in reviews[:800]:
        train_texts.append(review)
        train_labels.append(genre)
    for review in reviews[800:]:
        test_texts.append(review)
        test_labels.append(genre)

print("Train texts:", len(train_texts))
print("Test texts:", len(test_texts))
print("Labels:", sorted(set(train_labels)))

Train texts: 6400
Test texts: 1600
Labels: ['children', 'comics_graphic', 'fantasy_paranormal', 'history_biography', 'mystery_thriller_crime', 'poetry', 'romance', 'young_adult']


## 8. Baseline model using TF-IDF and Logistic Regression

This baseline is not the final model. It is included to show that the dataset is usable and to provide a simple comparison before DistilBERT fine-tuning.

In [10]:
vectorizer = TfidfVectorizer(max_features=20000)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, train_labels)
baseline_predictions = baseline_model.predict(X_test)

print(classification_report(test_labels, baseline_predictions))

                        precision    recall  f1-score   support

              children       0.65      0.65      0.65       200
        comics_graphic       0.82      0.72      0.77       200
    fantasy_paranormal       0.34      0.30      0.32       200
     history_biography       0.53      0.49      0.51       200
mystery_thriller_crime       0.51      0.54      0.52       200
                poetry       0.59      0.71      0.65       200
               romance       0.51      0.54      0.52       200
           young_adult       0.38      0.38      0.38       200

              accuracy                           0.54      1600
             macro avg       0.54      0.54      0.54      1600
          weighted avg       0.54      0.54      0.54      1600



## 9. Encode labels and tokenize text for DistilBERT

In [11]:
tokenizer = DistilBertTokenizerFast.from_pretrained(model_name)

unique_labels = sorted(set(train_labels))
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=max_length)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=max_length)

train_labels_encoded = [label2id[label] for label in train_labels]
test_labels_encoded = [label2id[label] for label in test_labels]

print("label2id:", label2id)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

label2id: {'children': 0, 'comics_graphic': 1, 'fantasy_paranormal': 2, 'history_biography': 3, 'mystery_thriller_crime': 4, 'poetry': 5, 'romance': 6, 'young_adult': 7}


## 10. Create PyTorch datasets

In [12]:
class GoodreadsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = GoodreadsDataset(train_encodings, train_labels_encoded)
test_dataset = GoodreadsDataset(test_encodings, test_labels_encoded)

## 11. Load pre-trained DistilBERT model

The model is loaded with the correct number of output labels for Goodreads genre classification.

In [13]:
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
).to(device_name)

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 12. Configure training with W&B logging

The key MLOps setting is `report_to="wandb"`, which sends training metrics to the W&B dashboard automatically.

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=16,
    learning_rate=learning_rate,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="wandb",
    run_name=wandb_run_name,
)

## 13. Define evaluation metrics

Accuracy and weighted F1 are returned, as required in the assignment.

In [15]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }

## 14. Fine-tune the model

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

/tmp/ipykernel_58/4113695406.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.245600,1.221331,0.565625,0.559615
2,1.030900,1.156692,0.586250,0.587114
3,0.716000,1.177005,0.588125,0.585548


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=1200, training_loss=1.0901119422912597, metrics={'train_runtime': 604.842, 'train_samples_per_second': 31.744, 'train_steps_per_second': 1.984, 'total_flos': 2543646198988800.0, 'train_loss': 1.0901119422912597, 'epoch': 3.0})

## 15. Final evaluation and explicit W&B metric logging

In [17]:
eval_results = trainer.evaluate()
print(eval_results)

wandb.log({
    "final/loss": eval_results["eval_loss"],
    "final/accuracy": eval_results["eval_accuracy"],
    "final/f1": eval_results["eval_f1"],
})

with open("eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 1.1566922664642334, 'eval_accuracy': 0.58625, 'eval_f1': 0.5871144378768375, 'eval_runtime': 14.7496, 'eval_samples_per_second': 108.477, 'eval_steps_per_second': 3.39, 'epoch': 3.0}


## 16. Save classification report and upload it as a W&B Artifact

In [18]:
prediction_output = trainer.predict(test_dataset)
preds = prediction_output.predictions.argmax(-1)

report = classification_report(
    test_labels_encoded,
    preds,
    target_names=[id2label[i] for i in sorted(id2label.keys())],
    output_dict=True,
)

with open("eval_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(classification_report(
    test_labels_encoded,
    preds,
    target_names=[id2label[i] for i in sorted(id2label.keys())]
))

artifact = wandb.Artifact("eval-report", type="evaluation")
artifact.add_file("eval_report.json")
artifact.add_file("eval_results.json")
wandb.log_artifact(artifact)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


                        precision    recall  f1-score   support

              children       0.72      0.70      0.71       200
        comics_graphic       0.83      0.76      0.79       200
    fantasy_paranormal       0.42      0.50      0.46       200
     history_biography       0.57      0.53      0.55       200
mystery_thriller_crime       0.52      0.62      0.57       200
                poetry       0.80      0.74      0.77       200
               romance       0.52      0.56      0.53       200
           young_adult       0.36      0.29      0.32       200

              accuracy                           0.59      1600
             macro avg       0.59      0.59      0.59      1600
          weighted avg       0.59      0.59      0.59      1600



<Artifact eval-report>

## 17. Save locally and push model to Hugging Face Hub

In [19]:
trainer.save_model(cached_model_directory_name)
tokenizer.save_pretrained(cached_model_directory_name)

create_repo(repo_id=repo_id, token=HF_TOKEN, private=False, exist_ok=True)
model.push_to_hub(repo_id, token=HF_TOKEN)
tokenizer.push_to_hub(repo_id, token=HF_TOKEN)

wandb.run.summary["huggingface_model"] = hf_model_url
wandb.run.summary["github_repository"] = github_url
wandb.run.summary["kaggle_notebook"] = kaggle_url
wandb.run.summary["wandb_dashboard"] = wandb_dashboard_url

print("Model uploaded successfully:", hf_model_url)

README.md: 0.00B [00:00, ?B/s]

Uploading...:   0%|          | 0.00/263M [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Model uploaded successfully: https://huggingface.co/hemantkumarsri/distilbert-goodreads-genres


## 18. Final links and finish W&B run

In [20]:
print("GitHub Repository:", github_url)
print("Kaggle Notebook:", kaggle_url)
print("Hugging Face Model:", hf_model_url)
print("W&B Dashboard:", wandb_dashboard_url)

wandb.finish()

GitHub Repository: https://github.com/hemantkumarsri/mlops-assignment2
Kaggle Notebook: https://www.kaggle.com/code/hemantkumarsri/mlops-assignment-2-fine-tuning-classification-k
Hugging Face Model: https://huggingface.co/hemantkumarsri/distilbert-goodreads-genres
W&B Dashboard: https://wandb.ai/hemantkumarsri-mlops/mlops-assignment2


eval/accuracy,▁▇█▇
eval/f1,▁███
eval/loss,█▁▃▁
eval/runtime,█▃▂▁
eval/samples_per_second,▁▆▇█
eval/steps_per_second,▁▆▇█
final/accuracy,▁
final/f1,▁
final/loss,▁
test/accuracy,▁
+10,...


## 19. Submission checklist

- GitHub repository is public.
- Kaggle notebook is public.
- Hugging Face model repository is public.
- W&B dashboard is public or accessible through the provided project link.
- Final report PDF includes all links and summarizes model choice, Kaggle setup, W&B tracking, evaluation results, and learnings.